# Personal AI Clone

This notebook builds a personal AI chatbot that not only answers questions based on your data but can also use tools to perform actions, like sending notifications or recording information.

## 1. Setup

Install dependencies, load environment variables, and set up API clients.

In [1]:
import os
import json
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

# Load environment variables
load_dotenv(override=True)

# Configure clients
openai_client = OpenAI()

# Global Configuration
KNOWLEDGE_BASE_DIR = "me/"
URL_LIST_FILE = os.path.join(KNOWLEDGE_BASE_DIR, "links.txt")

# Pushover Configuration
PUSHOVER_USER = os.getenv("PUSHOVER_USER")
PUSHOVER_TOKEN = os.getenv("PUSHOVER_TOKEN")
PUSHOVER_URL = "https://api.pushover.net/1/messages.json"

## 2. Data Ingestion

Load all documents from the `/me` folder to build the knowledge base context.

In [ ]:
def scrape_text_from_url(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        for item in soup(['script', 'style']): item.decompose()
        text = ' '.join(line.strip() for line in soup.get_text().splitlines() if line.strip())
        return text
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return ''

def load_knowledge_base(directory):
    full_context = []
    for filename in os.listdir(directory):
        if filename.endswith('.pdf'):
            path = os.path.join(directory, filename)
            try:
                reader = PdfReader(path)
                pdf_text = ''.join(page.extract_text() or '' for page in reader.pages)
                # Limit to first 4000 chars to prevent crashing context window
                truncated_text = pdf_text[:4000] 
                full_context.append(f'--- Content from {filename} ---\n{truncated_text}')
                print(f'Loaded PDF: {filename}')
            except Exception as e:
                print(f'Error reading {filename}: {e}')
    if os.path.exists(URL_LIST_FILE):
        with open(URL_LIST_FILE, 'r') as f:
            for url in (line.strip() for line in f if line.strip()):
                print(f'Scraping: {url}')
                if (scraped_text := scrape_text_from_url(url)):
                    full_context.append(f'--- Content from {url} ---\n{scraped_text}')
    return '\n\n'.join(full_context)

knowledge_context = load_knowledge_base(KNOWLEDGE_BASE_DIR)
print("\n--- Knowledge Base loading complete. ---")

Loaded PDF: resume.pdf
Scraping: https://sama-ndari.github.io/Portfolio/

--- Knowledge Base loading complete. ---


## 3. Tool Definitions and Implementation

Define the tools the AI can use, their JSON schemas, and the Python functions that execute them.

In [3]:
def push(message):
    """Sends a push notification via Pushover."""
    if not PUSHOVER_USER or not PUSHOVER_TOKEN:
        print("Pushover credentials not set. Skipping notification.")
        return
    print(f"Sending push notification: {message}")
    payload = {"user": PUSHOVER_USER, "token": PUSHOVER_TOKEN, "message": message}
    try:
        requests.post(PUSHOVER_URL, data=payload)
    except requests.RequestException as e:
        print(f"Failed to send push notification: {e}")

def record_user_details(email, name="Not provided", notes="Not provided"):
    """Records user details and sends a notification."""
    push(f"New contact inquiry from {name} ({email}). Notes: {notes}")
    return {"status": "Details recorded successfully."}

def record_unknown_question(question):
    """Records a question the chatbot could not answer."""
    push(f"An unanswerable question was asked: '{question}'")
    return {"status": "Question recorded."}

tools = [
    {
        "type": "function",
        "function": {
            "name": "record_user_details",
            "description": "Use this tool to record a user's contact details if they want to get in touch.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string", "description": "The user's email address."},
                    "name": {"type": "string", "description": "The user's name."},
                    "notes": {"type": "string", "description": "Contextual notes from the conversation."}
                },
                "required": ["email"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "record_unknown_question",
            "description": "Use this tool to record a question that could not be answered from the provided context.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string", "description": "The exact question that could not be answered."}
                },
                "required": ["question"]
            }
        }
    }
]

## 4. System Prompt and Tool Handler

Update the system prompt to instruct the AI on tool usage and create a handler to execute tool calls.

In [4]:
name = "Jules Cesar Junior NDAYISENGA"

system_prompt = f"""You are a helpful AI assistant acting as {name},
 representing him on his personal website. Your persona is professional,
  confident, and thoughtful.\n\nYour primary goal is to answer questions about {name}'s career,
   background, and skills using the provided context. Speak in the first person ('I').
   \n\n**Rules & Capabilities:**\n
    1.  **Grounded Answers:** Base your answers strictly on the context provided below.
     Do not invent information.\n
    2.  **Tool for Unknown Questions:** If you cannot answer a question from the context,
     you MUST use the `record_unknown_question` tool.
      Then, inform the user that you don't have the information.\n
    3.  **Tool for Contact:** If the user expresses interest in getting in touch, 
     ask for their email, name, and any relevant notes, 
     then use the `record_user_details` tool to capture this information.\n
    4.  **Polite Refusal:** Do not answer questions that are unrelated to {name}'s professional life.
     Politely decline and steer the conversation back to professional topics.
     \n\n--- CONTEXT ---\n{knowledge_context}\n--- END CONTEXT ---""".strip()

def handle_tool_calls(tool_calls):
    """Executes tool calls requested by the model."""
    tool_outputs = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Executing tool: {tool_name} with args: {arguments}")
        
        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)
        else:
            result = {"error": f"Tool '{tool_name}' not found."}
            
        tool_outputs.append({
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": tool_name,
            "content": json.dumps(result)
        })
    return tool_outputs

## 5. Gradio Chat Interface with Tool-Calling Loop

The main chat function now includes a loop to handle multi-step tool calls.

In [ ]:
def chat(message, history):
    # Format history for the API
    formatted_history = []
    for user_msg, ai_msg in history:
        formatted_history.append({"role": "user", "content": user_msg})
        formatted_history.append({"role": "assistant", "content": ai_msg})
    
    messages = [
        {"role": "system", "content": system_prompt},
        *formatted_history,
        {"role": "user", "content": message}
    ]

    # Loop to handle tool calls
    while True:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls

        if not tool_calls:
            # No tool calls, return the final response
            return response_message.content
        
        # Execute tool calls and get results
        tool_outputs = handle_tool_calls(tool_calls)
        
        # Append the assistant's response and tool outputs to the message history
        messages.append(response_message)
        messages.extend(tool_outputs)

gr.ChatInterface(
    chat,
    title="Personal AI Clone with Tools",
    description="Ask me about my professional background or ask to get in touch.",
    examples=["What is your experience with AI?", "I'd like to discuss a project with you, can I leave my email?"]
).launch(share=True)

/Users/samandari/Documents/Personal/AI/II.Udemy - The Complete Agentic AI Engineering Course (2025) 2025-6/Files/Projects/agents/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://37eba50e92ba753eb4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Executing tool: record_unknown_question with args: {'question': 'do u have a pattent?'}
Sending push notification: An unanswerable question was asked: 'do u have a pattent?'
Executing tool: record_user_details with args: {'email': 'sam@gmail.com', 'name': 'Samandari', 'notes': 'new to LLMs'}
Sending push notification: New contact inquiry from Samandari (sam@gmail.com). Notes: new to LLMs
